# 📦 01 — Dataset Exploration
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives
1. Understand the **structure and scale** of the SmartMine Unified Dataset.
2. Compute per-split **image and annotation counts**.
3. Analyse **class distribution and imbalance**.
4. Visualise **bounding box geometry** (size, position).
5. Examine **sample images** with ground-truth annotations.
6. Produce **publication-quality figures** saved to `outputs/images/`.

---

| Property | Value |
|---|---|
| Dataset | SmartMine Unified Dataset |
| License | MIT / CC BY 4.0 |
| Format | YOLOv8 (cx cy w h — normalised) |
| Pre-processing | 640×640 stretch, auto-orient, 5× augmentation |
| Splits | train / valid / test |

## 1. Setup & Imports

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Resolve project root by walking up until 'src/' is found
_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2

from src.ppe_detection.utils import (
    ensure_dirs, OUTPUTS_DIR,
    TRAIN_IMAGES, TRAIN_LABELS,
    VALID_IMAGES, VALID_LABELS,
    TEST_IMAGES, TEST_LABELS,
    CLASS_NAMES, CLASS_COLORS,
)
from src.ppe_detection.dataset_loader import (
    load_dataset_stats, stats_to_dataframe, class_distribution_dataframe,
)
from src.ppe_detection.visualization import (
    plot_split_sizes, plot_class_distribution, plot_sample_images,
)

ensure_dirs()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("All imports successful.")

## 2. Dataset Overview

We load statistics for all three splits. Each split contains an `images/` folder
and a corresponding `labels/` folder in YOLO format.

In [ ]:
stats      = load_dataset_stats()
summary_df = stats_to_dataframe(stats)

# Print rich summary
print("=" * 55)
print("  SMARTMINE UNIFIED DATASET — SUMMARY")
print("=" * 55)
print(summary_df.to_string())
print("-" * 55)
print(f"  TOTAL images      : {summary_df['images'].sum():,}")
print(f"  TOTAL annotations : {summary_df['annotations'].sum():,}")
avg_ann = summary_df['annotations'].sum() / summary_df['images'].sum()
print(f"  Avg annotations   : {avg_ann:.1f} per image")
print("=" * 55)

In [ ]:
plot_split_sizes(summary_df, save_path=OUTPUTS_DIR / "images" / "split_sizes.png")

## 3. Class Distribution

The dataset contains **32 classes** covering compliant PPE, violations, persons, vehicles, and environment.

| ID | Clase | Rol |
|---|---|---|
| 0 | person | 👤 Trabajador genérico |
| 1 | person_con_casco | ✅ Casco presente |
| 2 | person_sin_casco | ❌ Falta casco |
| 3 | person_con_chaleco | ✅ Chaleco presente |
| 4 | person_sin_chaleco | ❌ Falta chaleco |
| 5 | person_con_guantes | ✅ Guantes presentes |
| 6 | person_sin_guantes | ❌ Falta guantes |
| 7 | person_con_lentes | ✅ Lentes presentes |
| 8 | person_sin_lentes | ❌ Falta lentes |
| 9 | person_con_respirador | ✅ Respirador presente |
| 10 | person_sin_respirador | ❌ Falta respirador |
| 11 | person_ropa_reflectiva | ✅ Ropa reflectiva |
| 12 | person_sin_ropa_reflectiva | ❌ Falta ropa reflectiva |
| 13 | mask | 😷 Barbijo |
| 14–24 | camioneta / minibus / volquete / camion / excavadora / retro_excavadora / cargador_frontal / motoniveladora / tractor / rodillo / cisterna_agua | 🚛 Vehículos mineros |
| 25–28 | safety_cone / senalizacion / hardhat / safety_vest | 🦺 Seguridad |
| 29–31 | animal / polvo / machinery | 🌍 Entorno |

> **Class imbalance** is expected: `person` and `machinery` dominate.
> YOLO handles this automatically via focal loss, but we document it here.

In [ ]:
dist_df = class_distribution_dataframe(stats)

print("CLASS DISTRIBUTION ACROSS SPLITS")
print("=" * 60)
print(dist_df.to_string())
print()

# Highlight PPE-critical classes
ppe_ids = [0, 2, 4, 5, 7]
ppe_df  = dist_df[dist_df.index.isin(ppe_ids)]
print("PPE-CRITICAL CLASSES ONLY")
print("-" * 60)
print(ppe_df.to_string())

In [ ]:
plot_class_distribution(
    dist_df.reset_index(),
    save_path=OUTPUTS_DIR / "images" / "class_distribution.png",
)

In [ ]:
# Pie chart — proportion of each class in train split
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: all classes
train_counts = dist_df["train"].values
labels       = dist_df["class_name"].values
colors_list  = [
    tuple(c / 255 for c in CLASS_COLORS.get(i, (150,150,150)))
    for i in dist_df.index
]
axes[0].pie(
    train_counts, labels=labels, colors=colors_list,
    autopct="%1.1f%%", startangle=140, pctdistance=0.82,
)
axes[0].set_title("Train Set — Class Share", fontsize=13, fontweight="bold")

# Right: PPE classes only
ppe_counts = ppe_df["train"].values
ppe_labels = ppe_df["class_name"].values
ppe_colors = [
    tuple(c / 255 for c in CLASS_COLORS.get(i, (150,150,150)))
    for i in ppe_df.index
]
axes[1].pie(
    ppe_counts, labels=ppe_labels, colors=ppe_colors,
    autopct="%1.1f%%", startangle=140, pctdistance=0.82,
)
axes[1].set_title("PPE Classes Only — Train Share", fontsize=13, fontweight="bold")

plt.suptitle("Class Proportions in Training Set", fontsize=15, y=1.01)
plt.tight_layout()
save_path = OUTPUTS_DIR / "images" / "class_pie_charts.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

## 4. Bounding Box Geometry Analysis

Understanding bbox size and position helps detect annotation errors and anticipate
what scale of objects the model needs to detect.

In [ ]:
def load_all_bboxes(labels_dir: Path, max_files: int = 500) -> np.ndarray:
    """Load bounding boxes (cx, cy, w, h) from up to max_files label files."""
    boxes = []
    files = sorted(labels_dir.glob("*.txt"))[:max_files]
    for txt in files:
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                boxes.append([int(parts[0])] + [float(p) for p in parts[1:]])
    return np.array(boxes) if boxes else np.empty((0, 5))

boxes = load_all_bboxes(TRAIN_LABELS, max_files=500)
print(f"Loaded {len(boxes):,} bounding boxes from 500 train label files")

w_vals = boxes[:, 3]  # normalised width
h_vals = boxes[:, 4]  # normalised height
cx_vals = boxes[:, 1]
cy_vals = boxes[:, 2]

print(f"\nBBox width  — mean: {w_vals.mean():.3f}  median: {np.median(w_vals):.3f}  max: {w_vals.max():.3f}")
print(f"BBox height — mean: {h_vals.mean():.3f}  median: {np.median(h_vals):.3f}  max: {h_vals.max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Width & Height distributions
axes[0].hist(w_vals, bins=50, color="#4CAF50", alpha=0.8, label="width")
axes[0].hist(h_vals, bins=50, color="#2196F3", alpha=0.7, label="height")
axes[0].set_xlabel("Normalised size")
axes[0].set_ylabel("Count")
axes[0].set_title("BBox Width & Height Distribution")
axes[0].legend()

# Center position heatmap (where objects appear in image)
axes[1].hist2d(cx_vals, cy_vals, bins=40, cmap="YlOrRd")
axes[1].set_xlabel("Centre X (normalised)")
axes[1].set_ylabel("Centre Y (normalised)")
axes[1].set_title("Object Position Heatmap")
axes[1].invert_yaxis()

# Aspect ratio (w/h)
ar = w_vals / np.maximum(h_vals, 1e-6)
axes[2].hist(ar, bins=50, color="#FF9800", alpha=0.8)
axes[2].axvline(1.0, color="red", linestyle="--", label="square")
axes[2].set_xlabel("Aspect Ratio (w/h)")
axes[2].set_title("Bounding Box Aspect Ratio")
axes[2].legend()

plt.suptitle("Bounding Box Geometry — Training Set (500 files sample)", fontsize=13)
plt.tight_layout()
geo_path = OUTPUTS_DIR / "images" / "bbox_geometry.png"
plt.savefig(geo_path, dpi=150)
plt.show()
print(f"Saved → {geo_path}")

## 5. Annotations Per Image

Knowing how many objects appear per image helps set expectations for inference speed and
informs whether mosaic augmentation will be useful.

In [ ]:
ann_counts = []
for txt in sorted(TRAIN_LABELS.glob("*.txt"))[:500]:
    lines = [l for l in txt.read_text().splitlines() if l.strip()]
    ann_counts.append(len(lines))

ann_arr = np.array(ann_counts)
print(f"Annotations per image (train sample, n=500)")
print(f"  Min    : {ann_arr.min()}")
print(f"  Max    : {ann_arr.max()}")
print(f"  Mean   : {ann_arr.mean():.1f}")
print(f"  Median : {np.median(ann_arr):.0f}")
print(f"  Std    : {ann_arr.std():.1f}")
print(f"  Images with 0 ann: {(ann_arr == 0).sum()} (background images)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ann_arr, bins=range(0, ann_arr.max() + 2), color="#9C27B0", alpha=0.8, edgecolor="white")
ax.set_xlabel("Annotations per image")
ax.set_ylabel("Count")
ax.set_title("Annotations per Image Distribution (train, 500 sample)")
plt.tight_layout()
ann_path = OUTPUTS_DIR / "images" / "annotations_per_image.png"
plt.savefig(ann_path, dpi=150)
plt.show()
print(f"Saved → {ann_path}")

## 6. Sample Images with Ground-Truth Annotations

Visual inspection of the raw dataset with YOLO labels drawn as bounding boxes.
Each colour corresponds to a class (see `src/ppe_detection/utils.py`).

In [ ]:
plot_sample_images(
    images_dir=TRAIN_IMAGES,
    labels_dir=TRAIN_LABELS,
    n=12,
    save_path=OUTPUTS_DIR / "images" / "sample_annotations.png",
    seed=42,
)

In [ ]:
# Validation split samples
plot_sample_images(
    images_dir=VALID_IMAGES,
    labels_dir=VALID_LABELS,
    n=6,
    save_path=OUTPUTS_DIR / "images" / "sample_valid.png",
    seed=7,
)

## 7. Key Findings

| Finding | Value |
|---|---|
| Total images | 5,785 |
| Total annotations (all splits) | ~38,352 |
| Most frequent class | person (9,872 annotations) |
| Least frequent class | mask (1,700 annotations) |
| Avg annotations / image | ~13.7 |
| Background images (empty labels) | Present in valid/test |
| Dominant bbox size | Small-medium (mean w≈0.08, h≈0.14) |
| Object position | Distributed across full image |

> **Imbalance note:** `person` is 5× more frequent than `mask`.
> YOLOv8 uses focal loss which mitigates this, but rare classes
> (`mask`, `machinery`) may show lower recall.

## 8. Conclusions & Next Steps

**Conclusions:**
- The dataset is clean, pre-split, and ready for training.
- Violation classes (`person_sin_casco`, `person_sin_chaleco`) have sufficient samples (~2k–4k).
- Objects are mostly small-to-medium relative to image size → 640px input is appropriate.
- Some background images exist → useful for suppressing false positives during training.

**Next:** `02_dataset_validation.ipynb` — full integrity check on all 5,785 images.